In [30]:
print("all ok")

all ok


In [31]:
import sys, os

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
sys.path.append(project_root)

print("Project root added to path:", project_root)

Project root added to path: d:\gen ai projects\langchain and its projects\automated-research-report-generation


In [32]:
from research_and_analyst.utils.model_loader import ModelLoader

In [33]:
from langgraph.graph import StateGraph,START,END
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage
from langgraph.checkpoint.memory import MemorySaver

In [34]:
model_loader = ModelLoader()

{"timestamp": "2026-04-09T09:18:55.822099Z", "level": "info", "event": "Initializing ApiKeyManager"}
{"timestamp": "2026-04-09T09:18:55.823383Z", "level": "info", "event": "OPENAI_API_KEY loaded successfully from environment"}
{"timestamp": "2026-04-09T09:18:55.825790Z", "level": "info", "event": "GOOGLE_API_KEY loaded successfully from environment"}
{"timestamp": "2026-04-09T09:18:55.826789Z", "level": "info", "event": "GROQ_API_KEY loaded successfully from environment"}
{"path": "D:\\gen ai projects\\langchain and its projects\\automated-research-report-generation\\research_and_analyst\\config\\configuration.yaml", "keys": ["astra_db", "embedding_model", "retriever", "llm"], "timestamp": "2026-04-09T09:18:55.834570Z", "level": "info", "event": "Configuration loaded successfully"}
{"config_keys": ["astra_db", "embedding_model", "retriever", "llm"], "timestamp": "2026-04-09T09:18:55.836640Z", "level": "info", "event": "YAML configuration loaded successfully"}


In [35]:
llm = model_loader.load_llm()

{"provider": "openai", "model": "gpt-4o", "timestamp": "2026-04-09T09:18:56.729839Z", "level": "info", "event": "Loading LLM"}
{"provider": "openai", "model": "gpt-4o", "timestamp": "2026-04-09T09:18:56.738920Z", "level": "info", "event": "LLM loaded successfully"}


In [36]:
llm.invoke("hi").content

HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


'Hello! How can I assist you today?'

In [37]:
from typing import List
from typing_extensions import TypedDict
from pydantic import BaseModel, Field

In [38]:
class Analyst(BaseModel):
    name: str = Field(description="Name of the analyst.")
    role: str = Field(description="Role of the analyst in the context of the topic.")
    affiliation: str = Field(description="Primary affiliation of the analyst.")
    description: str = Field(description="Description of the analyst focus, concerns, and motives.")
    
    @property
    def persona(self) -> str:
        return f"Name: {self.name}\nRole: {self.role}\nAffiliation: {self.affiliation}\nDescription: {self.description}\n"
    

In [39]:
analyst = Analyst(
    name="vikas jangid",
    role="genai eng",
    affiliation="AI research Lab",
    description="I am a genai eng as well as mentor"
)

In [40]:
print(analyst.name)

vikas jangid


In [41]:
print(analyst.persona)

Name: vikas jangid
Role: genai eng
Affiliation: AI research Lab
Description: I am a genai eng as well as mentor



In [42]:

class Perspectives(BaseModel):
       analysts: List[Analyst] = Field(description="Comprehensive list of analysts with their roles and affiliations.")

In [43]:
class GenerateAnalystState(TypedDict):
    topic: str
    max_analysts: int
    human_analyst_feedback: str
    analysts: List[Analyst]

In [46]:

Analyst(
        name="Dr. Neha Patel",
        role="Medical Data Scientist",
        affiliation="Stanford Medicine",
        description="Focuses on predictive models for patient outcomes."
    ),

(Analyst(name='Dr. Neha Patel', role='Medical Data Scientist', affiliation='Stanford Medicine', description='Focuses on predictive models for patient outcomes.'),)

In [47]:
analyst_instructions="""You are tasked with creating a set of AI analyst personas. Follow these instructions carefully:

1. First, review the research topic:
{topic}
        
2. Examine any editorial feedback that has been optionally provided to guide creation of the analysts: 
        
{human_analyst_feedback}
    
3. Determine the most interesting themes based upon documents and / or feedback above.
                    
4. Pick the top {max_analysts} themes.

5. Assign one analyst to each theme."""

In [48]:
def create_analyst(state: GenerateAnalystState):
    """it is creating my analyst"""
    topic = state["topic"]
    max_analysts = state["max_analysts"]
    human_analyst_feedback = state.get("human_analyst_feedback","")

    structured_llm=llm.with_structured_output(Perspectives)

    system_message=analyst_instructions.format(topic=topic,
                                max_analysts=max_analysts,
                                human_analyst_feedback=human_analyst_feedback)
    
    analysts=structured_llm.invoke([SystemMessage(content=system_message)]+[HumanMessage(content="Generate the set of analysts")])

    return {"analysts": analysts.analysts}




In [49]:
create_analyst(
    {'topic': 'health',
    'max_analysts': 2,
    'human_analyst_feedback': 'give the real info'}
    )

HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


{'analysts': [Analyst(name='Dr. Emily Chen', role='Public Health Analyst', affiliation='World Health Organization', description='Dr. Emily Chen focuses on global health trends, particularly the impact of pandemics on public health systems. Her primary concern is understanding how health policies can be optimized to improve resilience against future health crises. She is motivated by the need to ensure equitable access to healthcare resources worldwide.'),
  Analyst(name='Dr. Raj Patel', role='Nutrition and Lifestyle Researcher', affiliation='Harvard T.H. Chan School of Public Health', description='Dr. Raj Patel specializes in the intersection of nutrition, lifestyle, and chronic disease prevention. His research is centered on how dietary patterns and physical activity influence long-term health outcomes. He is driven by the goal of reducing the prevalence of lifestyle-related diseases through evidence-based interventions and public education.')]}

In [9]:
def human_feedback(state):
    """human feedback"""
    pass

In [ ]:
def should_continue(state):
    """should continue"""
    pass

In [13]:
from IPython.display import Image, display

In [14]:
builder=StateGraph()

TypeError: StateGraph.__init__() missing 1 required positional argument: 'state_schema'

In [ ]:
builder.add_node()
builder.add_node()

NameError: name 'builder' is not defined

In [15]:
builder.edge(START,create_analyst)
builder.add_edge("create_analyst","human_feedback")
builder.conditional_edges("human_feedback",
                          should_continue,
                          ["create_analyst",
                           END])

NameError: name 'builder' is not defined

In [16]:
memory = MemorySaver()

In [ ]:
graph=builder.compile(interrupt_before=["human_feedback"], checkpointer=memory)

NameError: name 'builder' is not defined

In [ ]:
display(Image(graph.get_graph(xray=1).draw_mermaid_pnd()))